# CrimeDataBD: Multi-Class Crime Prediction Final ML Project

This Colab notebook is prepared for the final project demonstration.  
It keeps the previous workflow: dataset loading, EDA, preprocessing, train-test split, balancing, Logistic Regression, and Random Forest.  
Then it adds more model exploration requested by sir: **Decision Tree, Extra Trees, AdaBoost, Gradient Boosting, KNN, and XGBoost**.

**Target column:** `crime`  
**Dataset file:** `Bangladesh-Crime-Dataset.csv`


## 0) Install Required Libraries

Run this cell first in Google Colab.  
`imbalanced-learn` is used for SMOTENC and RandomUnderSampler.  
`xgboost` is used for the XGBoost model.


In [ ]:
# Install required libraries for Google Colab
!pip -q install imbalanced-learn xgboost

## 1) Load Dataset

Upload `Bangladesh-Crime-Dataset.csv` in Colab before running this cell.  
If the file is not found, the code will ask you to upload it.


In [ ]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Define dataset path
DATA_PATH = "Bangladesh-Crime-Dataset.csv"

# In Google Colab, if the CSV is not already uploaded, this will ask you to upload it.
if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Please upload Bangladesh-Crime-Dataset.csv")
    uploaded = files.upload()

# Load dataset
df = pd.read_csv(DATA_PATH)

# Show original shape
print("Original shape of dataset:", df.shape)

# Display first 5 rows
display(df.head())

# Drop unnecessary column if present
# Many CSV files contain an index column named 'Unnamed: 0'. It is not useful for ML.
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Show shape after dropping unnecessary column
print("Shape after dropping unnecessary column:", df.shape)

## 2) Missing Values and Class Distribution

This step helps us understand the dataset quality and target class balance.


In [ ]:
# Define target column
target = "crime"

# Check missing values
print("Total missing values in the dataset:", df.isna().sum().sum())

# Show number of unique crime classes
print("\nNumber of unique crime classes:", df[target].nunique())

# Show class distribution
print("\nClass distribution of the target variable:")
print(df[target].value_counts())

## 3) Visualize Class Distribution

Pie chart and bar chart help explain class imbalance during viva.


In [ ]:
# Count samples in each class
counts = df[target].value_counts()

# Plot pie chart
plt.figure(figsize=(7, 7))
plt.pie(
    counts.values,
    labels=counts.index.astype(str),
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Crime Class Distribution (Pie Chart)")
plt.tight_layout()
plt.show()

# Plot bar chart
plt.figure(figsize=(8, 5))
counts.plot(kind="bar")
plt.title("Crime Class Distribution (Bar Chart)")
plt.xlabel("Crime Type")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4) Feature and Target Separation

`X` contains input features.  
`y` contains the target crime class.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Separate features and target
X = df.drop(columns=[target])
y = df[target]

# Define categorical columns based on the dataset
cat_cols = [
    "incident_weekday",
    "weekend",
    "part_of_the_day",
    "incident_place",
    "incident_district",
    "incident_division",
    "season",
    "weather_code"
]

# Keep only categorical columns that actually exist in the dataset
cat_cols = [col for col in cat_cols if col in X.columns]

# Define numerical columns
num_cols = [col for col in X.columns if col not in cat_cols]

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)

## 5) Preprocessing and Train-Test Split

Categorical features are converted using OneHotEncoder.  
Numerical features are scaled using StandardScaler.  
The test set is kept original for honest evaluation.


In [ ]:
# Create preprocessing pipeline
# OneHotEncoder converts categorical text values into numeric columns.
# StandardScaler scales numeric columns so distance-based models like KNN work better.
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ],
    remainder="drop"
)

# Train-test split
# stratify=y keeps the class ratio similar in train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

## 6) Balance Only the Training Data

The dataset is imbalanced, so we balance the training data only.  
We do **not** balance the test data because test data should represent real-world distribution.


In [ ]:
from collections import Counter
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTENC

# Get categorical column indices for SMOTENC
cat_indices = [X.columns.get_loc(col) for col in cat_cols]

# Show original training class distribution
print("Original training class distribution:")
print(Counter(y_train))

# Step 1: undersample only classes that are above 1200
# This reduces majority classes so they do not dominate the model.
under_strategy = {
    "murder": 1200,
    "bodyfound": 1200
}

# Keep only labels that exist and have enough samples for undersampling
under_strategy = {k: v for k, v in under_strategy.items() if k in y_train.value_counts().index and y_train.value_counts()[k] >= v}

undersampler = RandomUnderSampler(
    sampling_strategy=under_strategy,
    random_state=40
)

X_train_under, y_train_under = undersampler.fit_resample(X_train, y_train)

print("\nClass distribution after undersampling:")
print(Counter(y_train_under))

# Step 2: oversample smaller classes to 1200
# SMOTENC creates synthetic samples and supports categorical features.
over_strategy = {
    "rape": 1200,
    "assault": 1200,
    "kidnap": 1200,
    "robbery": 1200
}

# Keep only labels that exist and need oversampling
current_counts = y_train_under.value_counts()
over_strategy = {k: v for k, v in over_strategy.items() if k in current_counts.index and current_counts[k] < v}

smote_nc = SMOTENC(
    categorical_features=cat_indices,
    sampling_strategy=over_strategy,
    random_state=40
)

X_train_resampled, y_train_resampled = smote_nc.fit_resample(X_train_under, y_train_under)

print("\nFinal balanced training class distribution:")
print(Counter(y_train_resampled))

## 7) Evaluation Helper Function

This function avoids repeating the same evaluation code for every model.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Store all results, trained models, and predictions
results = []
trained_models = {}
model_predictions = {}

def evaluate_model(model_name, model, X_train_data, y_train_data, X_test_data, y_test_data):
    """
    This function trains a model, predicts on test data, prints evaluation metrics,
    and stores results for final comparison.
    """
    print("=" * 70)
    print("Training Model:", model_name)
    print("=" * 70)

    # Train model
    model.fit(X_train_data, y_train_data)

    # Predict test data
    y_pred = model.predict(X_test_data)

    # Calculate metrics
    acc = accuracy_score(y_test_data, y_pred)
    macro_f1 = f1_score(y_test_data, y_pred, average="macro")
    weighted_f1 = f1_score(y_test_data, y_pred, average="weighted")

    # Print performance
    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("Weighted F1:", round(weighted_f1, 4))
    print("\nClassification Report:\n")
    print(classification_report(y_test_data, y_pred))

    # Store results
    results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1
    })

    trained_models[model_name] = model
    model_predictions[model_name] = y_pred

    return y_pred

## 8) Previous Model 1: Logistic Regression

This is your previous baseline model.  
It is useful because it is simple, fast, and easy to explain.


In [ ]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression pipeline
logreg_model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        penalty="l2",
        C=0.5
    ))
])

# Train and evaluate Logistic Regression
pred_lr = evaluate_model(
    "Logistic Regression",
    logreg_model,
    X_train_resampled,
    y_train_resampled,
    X_test,
    y_test
)

## 9) Previous Model 2: Random Forest

Random Forest is an ensemble of many decision trees.  
It usually performs well on tabular datasets.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest pipeline
rf_model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

# Train and evaluate Random Forest
pred_rf = evaluate_model(
    "Random Forest",
    rf_model,
    X_train_resampled,
    y_train_resampled,
    X_test,
    y_test
)

## 10) Additional Model Exploration Requested by Sir

This section adds more models without changing the previous workflow.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Define additional models
additional_models = {
    
    # Decision Tree:
    # Easy to explain because it works like if-else rules.
    # max_depth and min_samples help reduce overfitting.
    "Decision Tree": Pipeline(steps=[
        ("prep", preprocess),
        ("clf", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced",
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=2
        ))
    ]),
    
    # Extra Trees:
    # Similar to Random Forest, but adds more randomness while splitting trees.
    # It can reduce variance and sometimes gives better performance.
    "Extra Trees": Pipeline(steps=[
        ("prep", preprocess),
        ("clf", ExtraTreesClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=2,
            n_jobs=-1
        ))
    ]),
    
    # AdaBoost:
    # Trains weak learners one after another.
    # Each new learner focuses more on previous mistakes.
    "AdaBoost": Pipeline(steps=[
        ("prep", preprocess),
        ("clf", AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.5,
            random_state=42
        ))
    ]),
    
    # Gradient Boosting:
    # Builds models sequentially and tries to correct previous errors.
    # It is powerful but can be slower than simple models.
    "Gradient Boosting": Pipeline(steps=[
        ("prep", preprocess),
        ("clf", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ]),
    
    # KNN:
    # Predicts class based on nearest similar data points.
    # StandardScaler is important for KNN because it is distance-based.
    "KNN": Pipeline(steps=[
        ("prep", preprocess),
        ("clf", KNeighborsClassifier(
            n_neighbors=7,
            weights="distance"
        ))
    ])
}

# Train and evaluate each additional model
for model_name, model in additional_models.items():
    evaluate_model(
        model_name,
        model,
        X_train_resampled,
        y_train_resampled,
        X_test,
        y_test
    )

## 11) XGBoost Model

XGBoost is an advanced boosting algorithm and often performs very well on tabular data.  
For XGBoost, we encode target labels into numbers first.


In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encode target labels because XGBoost expects numeric class labels
label_encoder = LabelEncoder()
label_encoder.fit(y)

y_train_resampled_encoded = label_encoder.transform(y_train_resampled)
y_test_encoded = label_encoder.transform(y_test)

# XGBoost pipeline
xgb_model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softmax",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

print("=" * 70)
print("Training Model: XGBoost")
print("=" * 70)

# Train XGBoost using encoded target labels
xgb_model.fit(X_train_resampled, y_train_resampled_encoded)

# Predict encoded labels
xgb_pred_encoded = xgb_model.predict(X_test)

# Convert predictions back to original crime names
xgb_pred = label_encoder.inverse_transform(xgb_pred_encoded.astype(int))

# Evaluate XGBoost using original labels
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_macro_f1 = f1_score(y_test, xgb_pred, average="macro")
xgb_weighted_f1 = f1_score(y_test, xgb_pred, average="weighted")

print("Accuracy:", round(xgb_acc, 4))
print("Macro F1:", round(xgb_macro_f1, 4))
print("Weighted F1:", round(xgb_weighted_f1, 4))
print("\nClassification Report:\n")
print(classification_report(y_test, xgb_pred))

# Store XGBoost result
results.append({
    "Model": "XGBoost",
    "Accuracy": xgb_acc,
    "Macro F1": xgb_macro_f1,
    "Weighted F1": xgb_weighted_f1
})

trained_models["XGBoost"] = xgb_model
model_predictions["XGBoost"] = xgb_pred

## 12) Final Model Comparison Table

We compare all models using Accuracy, Macro F1, and Weighted F1.  
For this project, **Macro F1** is very important because the classes are imbalanced.


In [ ]:
# Create final comparison table
results_df = pd.DataFrame(results)

# If a cell is run multiple times, remove duplicate model names and keep latest result
results_df = results_df.drop_duplicates(subset=["Model"], keep="last")

# Sort by Macro F1 score
results_df = results_df.sort_values(by="Macro F1", ascending=False).reset_index(drop=True)

print("Final Model Comparison:")
display(results_df)

## 13) Model Comparison Graph

This graph makes the result easier to present to sir.


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(results_df["Model"], results_df["Macro F1"])
plt.title("Model Comparison Based on Macro F1 Score")
plt.xlabel("Machine Learning Models")
plt.ylabel("Macro F1 Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 14) Select the Best Model Automatically

The best model is selected using Macro F1 score.


In [ ]:
# Select best model based on Macro F1
best_model_name = results_df.iloc[0]["Model"]
best_prediction = model_predictions[best_model_name]
best_model = trained_models[best_model_name]

print("Best Model Based on Macro F1:", best_model_name)
print("Best Macro F1 Score:", round(results_df.iloc[0]["Macro F1"], 4))
print("Best Accuracy:", round(results_df.iloc[0]["Accuracy"], 4))

print("\nClassification Report of Best Model:\n")
print(classification_report(y_test, best_prediction))

## 15) Confusion Matrix for the Best Model

The confusion matrix shows which crime classes are correctly predicted and which classes are confused with each other.


In [ ]:
labels = sorted(y.unique())

cm_best = confusion_matrix(y_test, best_prediction, labels=labels)
disp_best = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=labels)
disp_best.plot(xticks_rotation=45)
plt.title(f"Confusion Matrix - Best Model: {best_model_name}")
plt.tight_layout()
plt.show()

## 16) Save the Best Model

The saved model contains both preprocessing and classifier, so it can be loaded later for prediction.


In [ ]:
# Save best model
joblib.dump(best_model, "best_crime_model_after_comparison.pkl")

# Save label encoder only if XGBoost is the best model
# This is useful because XGBoost was trained with encoded target labels.
if best_model_name == "XGBoost":
    joblib.dump(label_encoder, "xgboost_label_encoder.pkl")

print("Best model saved successfully as 'best_crime_model_after_comparison.pkl'")
print("Saved Model:", best_model_name)

## 17) Feature Importance for Tree-Based Best Models

Feature importance helps explain which features contributed most to prediction.  
This works for Random Forest, Decision Tree, Extra Trees, Gradient Boosting, and XGBoost.


In [ ]:
# Feature importance is available for many tree-based models.
# KNN and Logistic Regression do not have feature_importances_ in the same way.

try:
    final_clf = best_model.named_steps["clf"]
    feature_names = best_model.named_steps["prep"].get_feature_names_out()

    if hasattr(final_clf, "feature_importances_"):
        importances = final_clf.feature_importances_

        importance_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        }).sort_values(by="Importance", ascending=False)

        print("Top 15 Important Features:")
        display(importance_df.head(15))

        plt.figure(figsize=(10, 6))
        plt.barh(importance_df.head(15)["Feature"][::-1], importance_df.head(15)["Importance"][::-1])
        plt.title(f"Top 15 Feature Importances - {best_model_name}")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.show()
    else:
        print(f"{best_model_name} does not provide feature_importances_ directly.")

except Exception as e:
    print("Feature importance could not be shown.")
    print("Reason:", e)

## 18) Viva Notes

Use these points when sir asks why you used these models:

- **Logistic Regression:** Simple baseline model for classification.
- **Random Forest:** Ensemble of many decision trees; reduces overfitting compared to one tree.
- **Decision Tree:** Rule-based model; easy to understand and explain.
- **Extra Trees:** Similar to Random Forest but more randomized; often strong for tabular data.
- **AdaBoost:** Boosting method where each next learner focuses on previous mistakes.
- **Gradient Boosting:** Sequential boosting model that reduces previous errors step by step.
- **KNN:** Distance-based model that predicts using nearest similar data points.
- **XGBoost:** Advanced gradient boosting model; strong for structured/tabular datasets.

**Why Macro F1?**  
Macro F1 gives equal importance to every class, so it is better than only accuracy when classes are imbalanced.

**Why balance only training data?**  
We balance the training data so the model can learn minority classes. We keep the test data unchanged to evaluate the model honestly.
